# 01 — Data Audit & Pipeline Verification
Validate & Verify cleaned outputs (`master_food_dataset.csv` and `weather_regions_monthly.csv`), check zero nulls, and validate stitched series.


In [10]:
import pandas as pd

df_f = pd.read_csv('../data/processed/master_food_dataset.csv')
df_w = pd.read_csv('../data/processed/weather_regions_monthly.csv')

In [11]:
print('Master shape:', df_f.shape)
df_f.head()

Master shape: (7866, 26)


,filter_id,filter_en,filter_ar,category_en,category_ar,date,year,month,price,usd_buy_mean,...,regulated_inflation_mom,veg_fruit_inflation_mom,cbe_deposit_rate,cbe_lending_rate,cairo_temp_mean,nat_agro_temp_mean,nat_heatwave_days_avg,nat_cold_days_avg,nat_precip_total,nat_temp_anomaly
0,196,Onion,بصل,Vegetables & Fruits,خضر وفاكهة,2015-01-01,2015,1,3.55,7.2535,...,2.502,1.392,8.75,9.75,13.52,13.50,0.00,11.00,5.1,-0.34
1,196,Onion,بصل,Vegetables & Fruits,خضر وفاكهة,2015-02-01,2015,2,3.80,7.6004,...,3.309,4.319,8.75,9.75,14.46,14.37,0.00,7.67,13.4,-0.66
2,196,Onion,بصل,Vegetables & Fruits,خضر وفاكهة,2015-03-01,2015,3,3.90,7.6023,...,2.110,5.688,8.75,9.75,18.45,18.00,0.75,1.67,9.3,-0.01
3,196,Onion,بصل,Vegetables & Fruits,خضر وفاكهة,2015-04-01,2015,4,2.13,7.6040,...,-3.777,6.295,8.75,9.75,20.39,20.04,3.75,0.00,6.0,-1.88
4,196,Onion,بصل,Vegetables & Fruits,خضر وفاكهة,2015-05-01,2015,5,2.38,7.6040,...,0.080,6.156,8.75,9.75,25.28,24.87,9.75,0.00,4.8,-1.19


In [12]:
print('Weather regions shape:', df_w.shape)
df_w.head()

Weather regions shape: (552, 10)


,year,month,region,temp_mean,temp_max,temp_min,precip_sum,humidity_mean,heatwave_days,cold_days
0,2015,1,East_Delta_Sharqia,13.97,28.0,5.6,1.8,51.71,0,6
1,2015,1,Greater_Cairo,13.52,27.2,4.4,2.2,50.77,0,13
2,2015,1,Upper_Egypt_Minya,13.17,25.9,3.1,0.0,45.90,0,18
3,2015,1,West_Delta_Beheira,13.35,26.4,3.6,1.1,54.65,0,9
4,2015,2,East_Delta_Sharqia,14.56,30.0,4.6,4.1,52.21,0,8


In [16]:
print('master_food_dataset_nulls', df_f.isnull().sum().sum())
print('weather_regions_monthly_nulls', df_w.isnull().sum().sum())

master_food_dataset_nulls 0
weather_regions_monthly_nulls 0


In [18]:
print("Master dataset full-row duplication:", df_f.duplicated().sum())
print("Weather regions full-row duplication:", df_w.duplicated().sum())

print("Master primary-key duplication (filter_id, date):", df_f.duplicated(subset=["filter_id", "date"]).sum())
print("Weather primary-key duplication (region, year, month):", df_w.duplicated(subset=["region", "year", "month"]).sum())

Master dataset full-row duplication: 0
Weather regions full-row duplication: 0
Master primary-key duplication (filter_id, date): 0
Weather primary-key duplication (region, year, month): 0


In [15]:
# 2.(Structural & Domain Integrity Audit)

checks = [

    ("Invalid Prices (<= 0 EGP)", int((df_f["price"] <= 0).sum()), 0),
    ("Continuous Commodities Count", int(df_f["filter_id"].nunique()), 57),
    ("Stitched Farm Eggs Series (ID 9999)", int((df_f["filter_id"] == 9999).sum()), 138),
    ("Min Months per Commodity", int(df_f.groupby("filter_id")["date"].nunique().min()), 138),
    ("Max Months per Commodity", int(df_f.groupby("filter_id")["date"].nunique().max()), 138),

    ("USD Buy > USD Sell Anomalies", int((df_f["usd_buy_mean"] > df_f["usd_sell_mean"]).sum()), 0),
    ("Negative USD Volatility", int((df_f["usd_volatility"] < 0).sum()), 0),
    ("CBE Deposit > Lending Rate Anomalies", int((df_f["cbe_deposit_rate"] > df_f["cbe_lending_rate"]).sum()), 0),

    ("Weather temp_min > temp_mean", int((df_w["temp_min"] > df_w["temp_mean"]).sum()), 0),
    ("Weather temp_mean > temp_max", int((df_w["temp_mean"] > df_w["temp_max"]).sum()), 0),
    ("Negative Precipitation", int((df_w["precip_sum"] < 0).sum()), 0),
    ("Humidity Out of [0, 100]%", int(((df_w["humidity_mean"] < 0) | (df_w["humidity_mean"] > 100)).sum()), 0),
    ("Region Heatwave Days Out of [0, 31]", int(((df_w["heatwave_days"] < 0) | (df_w["heatwave_days"] > 31)).sum()), 0),
    ("Region Cold Days Out of [0, 31]", int(((df_w["cold_days"] < 0) | (df_w["cold_days"] > 31)).sum()), 0),
    ("Master Aggregated Precip < 0", int((df_f["nat_precip_total"] < 0).sum()), 0),
]

audit_df = pd.DataFrame(checks, columns=["Check Name", "Actual", "Expected"])
audit_df["Status"] = audit_df.apply(lambda r: "✅ PASS" if r["Actual"] == r["Expected"] else "❌ FAIL", axis=1)

print("=== 2. DOMAIN INTEGRITY AUDIT ===")
display(audit_df)

assert (audit_df["Status"] == "✅ PASS").all(), "Some data integrity checks failed!"

=== 2. DOMAIN INTEGRITY AUDIT ===


,Check Name,Actual,Expected,Status
0,Invalid Prices (<= 0 EGP),0,0,✅ PASS
1,Continuous Commodities Count,57,57,✅ PASS
2,Stitched Farm Eggs Series (ID 9999),138,138,✅ PASS
3,Min Months per Commodity,138,138,✅ PASS
4,Max Months per Commodity,138,138,✅ PASS
5,USD Buy > USD Sell Anomalies,0,0,✅ PASS
6,Negative USD Volatility,0,0,✅ PASS
7,CBE Deposit > Lending Rate Anomalies,0,0,✅ PASS
8,Weather temp_min > temp_mean,0,0,✅ PASS
9,Weather temp_mean > temp_max,0,0,✅ PASS
